# Customer Segmentation with K-Means

Exploratory analysis, feature engineering, scaling, and customer clustering.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import joblib
from pathlib import Path

In [ ]:
df = pd.read_csv("customer_segmentation.csv")
df.head(), df.shape, df.describe()

In [ ]:
df.info()
df.isnull().sum()
df.dropna(inplace=True)

In [ ]:
df['Age'] = 2026 - pd.to_numeric(df['Year_Birth'])
education_map = {'Graduation': 2, 'Master': 3, '2n Cycle': 1, 'Basic': 0, 'PhD': 4}
df['Education'] = df['Education'].map(education_map)
from sklearn.preprocessing import LabelEncoder
df['Marital_Status'] = LabelEncoder().fit_transform(df['Marital_Status'])
df['Total_kids'] = df['Kidhome'] + df['Teenhome']
spend_columns = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df['Total_spend'] = df[spend_columns].sum(axis=1)

In [ ]:
features = ['Age', 'Income', 'NumWebPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'Total_spend']
x = df[features].copy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

In [ ]:
inertias = []
for n_clusters in range(2, 10):
    candidate = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    candidate.fit(x_scaled)
    inertias.append(candidate.inertia_)
plt.plot(range(2, 10), inertias, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Clusters'] = kmeans.fit_predict(x_scaled)
df[['Clusters']].value_counts()

## Save trained artifacts

The following commands save the fitted model and scaler beside this notebook. The deployed Streamlit app expects these filenames.

In [ ]:
output_dir = Path('.')
joblib.dump(kmeans, output_dir / 'kmeans_model.pkl')
joblib.dump(scaler, output_dir / 'scaler.pkl')